# പാഠം 18: ക്രിപ്‌റ്റോഗ്രാഫിക് റെസീറ്റുകൾ ഉപയോഗിച്ച് AI എജന്റ്‌സ് സുരക്ഷിതമാക്കുക

## ഹാൻഡ്‌സ്-ഓൺ നോട്ട്‌بുക്ക്

ഈ നോട്ട്‌ബുക്ക് നാലു വ്യായാമങ്ങൾ വഴിയുള്ള വഴി കാണിക്കുന്നു:

1. എജന്റ് ടൂൾ കോളിനായി താങ്കളുടെ ആദ്യ റെസീറ്റ് ഒപ്പിടുക, അതിനെ സ്ഥിരീകരിക്കുക.
2. റെസീറ്റ് തട്ടി മാറ്റുക, തുടർന്ന് സ്ഥിരീകരണം പരാജയപ്പെടുന്നത് കാണുക.
3. മൂന്ന് റെസീറ്റ് ചൈനുണ്ടാക്കി, ചൈനിന്റെ അഖണ്ഡത ഉറപ്പാക്കുക.
4. ഒരു Microsoft Agent Framework ടൂൾ കോൾ റാപ്പ് ചെയ്ത് ഓരോ പ്രവർത്തനവും ഒരു റെസീറ്റ് പുറപ്പെടുവിക്കുക.

എല്ലാ ക്രിപ്‌റ്റോഗ്രാഫിക് പ്രിമിറ്റീവുകളും നന്നായി പരിപാലിച്ച ലൈബ്രറികളിൽ നിന്ന് ഇറക്കുമതി ചെയ്യുന്നു (`pynacl` Ed25519-ക്കിടയിൽ, `jcs` RFC 8785 canonical JSON-ക്കായി, Python സ്റ്റാൻഡേർഡ് ലൈബ്രറിയിൽ നിന്നുള്ള `hashlib` SHA-256-ക്കായി). റെസീറ്റ് ലാജിക്ക് സ്വയം വായിക്കാനും മാറ്റാനും കഴിയുന്ന സാധാരണ Python ആണ്.

സെല്ലുകൾ ക്രമത്തിൽ പ്രവർത്തിപ്പിക്കുക. ഓരോ വിഭാഗവും ചെറിയതും സ്വയംപര്യാപ്തവുമാണ്.


## ക്രമീകരണം

രണ്ട് ആശ്രിതങ്ങളും ഇൻസ്റ്റാൾ ചെയ്യുക. ഇരുയക്തികൂടളും (Apache-2.0 / MIT) അനുവാദകരമായ ലൈസൻസുകളാണ്.


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## സഹായക ഉപകരണങ്ങൾ

ഈ രണ്ട് സഹായികൾ ബേസ്64url എൻകോഡിംഗ് (പാഡിംഗ് ഇല്ലാതെ) കൂടാതെ എവിടെ നിന്നും വെക്കാവുന്ന ഒബ്ജക്റ്റുകളുടെ SHA-256 ഹാഷിംഗ് കൈകാര്യം ചെയ്യുന്നു. അവ ബാക്കി നോട്ട്‌ബുക്ക് വാങ്ങൽ ലജിക് താനെ കേന്ദ്രീകരിക്കാനും സഹായിക്കുന്നു.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## വിഭാഗം 1: നിങ്ങളുടെ ആദ്യ ഇൻവോയിസ് ഒപ്പിടുക

**Contoso Travel**ന്റെ നമ്മുടെ ഏജന്റ് ഒരു ഉപഭോക്താക്കായി സിഡ്‌നി മുതൽ ലോസ് ആഞ്ചലസ് വരെയുള്ള വിമാന യാത്രകൾ പരിശോധിച്ചു എന്ന് കണക്കാക്കൂ. വിശ്വസിക്കാതെ ഭാവി ഓഡിറ്റർ അത് സ്ഥിരീകരിക്കാനായി ഈ ടൂൾ കോൾ ഒപ്പിട്ട ഇൻവോയിസായി രേഖപ്പെടുത്താൻ നമുക്ക് വേണം.

### ഘട്ടം 1.1: ഒപ്പിടാ കീ സൃഷ്ടിക്കുക

ഉത്പാദനത്തിൽ, ഏജന്റിന്റെ ഒപ്പിടുന്ന കീ ഒരു ഹാർഡ്‌വെയർ സെക്യൂരിറ്റി മോഡ്യൂളിൽ (HSM), ആധാർ കീ വാൾട്ട്, അല്ലെങ്കിൽ സമാനമായ സംരക്ഷിത സംഭരണത്തിൽ നിര്‍വഹിച്ചു. ഈ പാഠത്തിനായി ഞങ്ങൾ ഒരു പുതിയ കീ മെമ്മറിയിൽ സൃഷ്ടിക്കുന്നു.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ഘട്ടം 1.2: രസീത് പേലോഡ് നിർമ്മിക്കുക

പേലോഡിൽ നാം രസീത് സ്ഥിരീകരിക്കേണ്ടതെല്ലാം ഉൾപ്പെടുന്നു: ആരാണ് പ്രവർത്തിച്ചത്, ഏത് ഉപകരണം, എന്ത് വാദങ്ങൾ ഉപയോഗിച്ച്, എന്താണ് മടങ്ങിവന്നത്, ഏത് നയത്തിലാണെന്ന്, 언제. നാം വാദങ്ങളും ഫലവും inline ഉൾപ്പെടുത്താതെ അത് ഹാഷ് ചെയ്യുന്നു, ഇതുവഴി രസീത് സങ്കീർണ്ണ ഉള്ളടക്കം മറയ്ക്കാൻ സഹായിക്കുന്നു.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### പടി 1.3: റിസീത് ഒപ്പുവെച്ച് ചേര്ക്കുക

മൂന്നു പടികൾ:

1. രണ്ട് നടപ്പികൾ സമാനമായ ലൊജിക്കൽ റിസീത് സൃഷ്ടിക്കുമ്പോൾ ബൈറ്റ്-ഒരുമയായ ബൈറ്റുകൾ ഉളവാക്കാൻ JCS ഉപയോഗിച്ച് പേലോഡ് കാനോണിക്കൽ ആക്കുക.
2. കാനോണിക്കൽ ബൈറ്റുകളെ SHA-256 ഉപയോഗിച്ച് ഹാഷ് ചെയ്യുക.
3. Ed25519 സ്വകാര്യ കീ ഉപയോഗിച്ച് ഹാഷ് ഒപ്പിടുക.

ഒപ്പു പിന്നീട് യഥാർത്ഥ പേലോഡിൽ ചേർക്കപ്പെടുകയും അന്തിമ റിസീത് ഉണ്ടാക്കുകയും ചെയ്യുന്നു.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ഘട്ടം 1.4: رسید ശരിയാണോയെന്ന് പരിശോധിക്കുക

പരിശോധന പ്രക്രിയയെ മറുപ്രവർത്തനമാക്കുന്നു. ഞങ്ങൾ ഒപ്പ് പിന്‍വലിച്ച്, കാനോണിക്കൽ ഹാഷ് വീണ്ടും കണക്കാക്കി, رسیدയിലെ പബ്ലിക് കിയിലെതിരെ ഒപ്പ് പരിശോധിക്കുന്നു.

ഈ പരിശോധന നടത്തുന്നതിനായി ഒരു ഓഡിറ്ററിന് ഞങ്ങളിലോ അറിവോ ആവശ്യമില്ല, صرف رسید തന്നെയാണ് വേണ്ടത്. സേവ്‌വീസ് വിളിക്കേണ്ടതില്ല, കീ ഡയറക്ടറി ചോദിക്കേണ്ടതില്ല, വിശ്വാസം ആവശ്യമില്ല.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

നിങ്ങൾക്ക് `Receipt is valid: True` കാണണം. ഏജന്റ് ആദ്യത്തെ ക്രിപ്റ്റോഗ്രാഫിക് ആയി ഒപ്പുവച്ച ഓഡിറ്റ് റെക്കോർഡ് സൃഷ്ടിച്ചിരിക്കുന്നു.


## ഭാഗം 2: സ്വീകരണം തട്ടിപ്പിടിക്കുക

സ്വീകരണങ്ങൾ തട്ടിപ്പു തെളിയിക്കാവുന്നതാണ് എന്നതാണ് മുഴുവൻ ആശയം. ഇതിന് തെളിവുകൾ കാണിക്കാം.

സ്വീകരണത്തിലെ ഒറ്റപ്പെട്ട ഒരു അക്ഷരം മാത്രമേ ഞങ്ങൾ മാറ്റൂ, പിന്നെ വാലിഡേഷൻ പരാജയപ്പെടുന്നത് കാണാം.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ഇപ്പോഴിന്ന് സംഭവിച്ചത് എന്താണ്?

`policy_id` മാറ്റിയപ്പോൾ, canonical ബൈറ്റുകൾ മാറി. ആ ബൈറ്റുകളുടെ SHA-256 ഹാഷ് മാറി. ഒറിജിനൽ ഹാഷിന്റെ മുകളിലുള്ള സിഗ്നേച്ചർ പുതിയ ഹാഷിനൊപ്പം പൊരുത്തപ്പെടുന്നില്ല. സ്ഥിരീകരണം ശരിയായ രീതിയിൽ `False` ആണെന്ന് തിരിച്ചുനൽകുന്നു.

ആക്രമകന് സ്വകാര്യ കീ ഇല്ലാതെ റെസീപ്പ് ന്റെ ഏതെങ്കിലും ഫീൽഡ് തിരുത്തിയാലും അത് സ്ഥിരീകരിക്കാനാകില്ല. സ്വകാര്യ കീ കീ വീട്(കീ വോൾട്ട്)ൽ വെച്ചിരിക്കുന്നതും, പെblico ഇതിന്റെ പബ്ലിക് കീ പ്രസിദ്ധീകരിച്ചിട്ടുള്ളതും ആയാൽ, അഴിച്ചുപണി മറയ്ക്കാൻ കഴിയില്ല.

നിങ്ങൾ തന്നെ പരീക്ഷിക്കാം: മുകളിൽ ഉള്ള സെല്ലിൽ `tool_name` അല്ലെങ്കിൽ `agent_id` അല്ലെങ്കിൽ `timestamp` മാറ്റി വീണ്ടും റൺ ചെയ്യുക. ഓരോ മാറ്റവും സാധുവായ റെസീപ്പ് ഉണ്ടാക്കില്ല.


## വിഭാഗം 3: റസീപ്പ്റ്റുകൾ കയ്റയിടുക

ഒരൊറ്റ റസീപ്പ് ഒരേ പ്രവർത്തനം സംരക്ഷിക്കുന്നു. അധികസംഖ്യയിലുള്ള ഏജന്റുമാർ പല പ്രവർത്തനങ്ങളും ചെയ്യുന്നു. മുഴുവൻ ക്രമത്തിലും തട്ടിപ്പിന്റെ അടയാളമുണ്ടാകാതിരിക്കാനായി, ഓരോ റസീപ്പ്റ്റും മുമ്പത്തെ റസീപ്പ്റ്റിന്റെ ഹാഷ് പുതിയത് റസീപ്പ്റ്റിന്റെ പേലോഡിൽ ഉൾക്കൊള്ളിച്ച് മുമ്പത്തെ റസീപ്പ്റ്റുമായി ബന്ധിപ്പിക്കുന്നു.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ഏതോ ഒരാൾ ഒരു റസീപ്പ് നീക്കംചെയ്യുകയോ കാര്യക്രമം മാറ്റുകയോ ചെയ്താൽ, കയ്റ ശരിയായി അതുകാര്യത്തിൽ തകരും. പിന്നീട് വരുന്ന ഏതെങ്കിലും റസീപ്പ്റ്റിന്റെ പരിശോധന പരാജയപ്പെടുന്നത് കാരണം അതിന്റെ `previous_receipt_hash` ആഗ്രഹിക്കപ്പെട്ട পূর্বവർത്തിയുടെ യഥാർത്ഥ ഹാഷുമായി ഇനി സൂചിപ്പിക്കുകയില്ല.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ഇപ്പോൾ ചെയിൻ സമ്പർക്കം മധ്യത്തിലുള്ള റസീത് കുത്തikkenട്രിച്ച് തകർത്തുവെച്ചു വീണ്ടും ശരിവെക്കൂ. തകർത്ത റസീത്ത് അതിന്റെ ഒപ്പ് പരിശോധനയിൽ പരാജയപ്പെടുന്നു, കൂടാതെ അടുത്ത റസീത്ത് ചേൻ-ലിങ്ക് പരിശോധനയിൽ പരാജയപ്പെടുന്നു (കാരണം അതിന്റെ `previous_receipt_hash` ഇനി മാറ്റിയ മധ്യത്തിലുള്ള റസീത്തിന്റെ ഹാഷുമായി പൊരുത്തപ്പെടുന്നില്ല).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ക്രമരീതി 0 ഇപ്പോഴും സ്ഥിരീകരിക്കുന്നു (അത് തിരുത്തപ്പെട്ടിട്ടില്ല, ആശ്രയിക്കേണ്ട മുമ്പത്തെ രേഖയുമില്ല). ക്രമരീതി 1-ന് അതിന്റെ ഒപ്പ് പരിശോധന പരാജയപ്പെടുന്നു കാരണം ഞങ്ങൾ `tool_args_hash` മാറ്റി. ക്രമരീതി 2-ന് ഉള്ള ഒണ്ണിത്തരം-ചങ്ങല പരിശോധന പരാജയപ്പെടുന്നു കാരണം അതിന്റെ `previous_receipt_hash` ആദ്യം (ഇപ്പോൾ തിരുത്തപ്പെട്ട) ക്രമരീതി 1-നെ അടിസ്ഥാനമാക്കി കണക്കാക്കിയിരുന്നു.

ഒരു ആക്രമിച്ചയാൾ മാറ്റം ചെയ്ത ക്രമരീതി 1 വീണ്ടും സൈൻ ചെയ്താലും (അവർക്ക് സ്വകാര്യ കീ ഇല്ലാതെ അത് ചെയ്യാൻ കഴിയില്ല), ക്രമരീതി 2-ലെ ഒണ്ണിത്തരം-ചങ്ങലവ്യത്യാസം മാറ്റം വെളിപ്പെടുത്തും. മാറ്റം മറയ്ക്കാൻ, ആക്രമിച്ചയാൾ മാറ്റം നടന്ന സ അടിസ്ഥാനം മുതൽ എല്ലാ ക്രമരീതികളും വീണ്ടും ഒപ്പ് നിർബന്ധമുണ്ട്, അതിന് സ്വകാര്യ കീ സ്വന്തമാകണം.


## ഭാഗം 4: ഒരു ഏജന്റ് ടൂൾ കോളിനെ രസീത് ഒപ്പ് കൂട്ടിയും പൊതിയുക

യഥാർത്ഥ വിന്യാസത്തിൽ, ഓരോ ഏജന്റ് രേഖാകർത്താവും `make_receipt` വിളിക്കണമെന്ന് ഓർക്കണമെന്ന് നിങ്ങൾക്ക് ആവശ്യമില്ല. ഓരോ ടൂൾ വിളിച്ചലിനും രസീത് ഒപ്പ് സ്വയമേവ തടിച്ചിരിക്കണം.

ഏറ്റവും ലളിതമായ മാതൃക ഇവിടെ കാണാം: ഏതൊരു കാളബിൾ ടൂൾ ഫംഗ്ഷനേയും സ്വീകരിച്ച് അതിന്റെ ഒരു രസീത് പുറത്തിറക്കുന്ന പതിപ്പായി തിരിച്ചുനൽകുന്ന ഒരു റാപ്പർ ക്ലാസ്. ഇത് മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിമ്വർക്കും (`agent_framework.foundry`) അടക്കം ഏതൊരു ഏജന്റ് ഫ്രെയിംവർക്കിനും അനുയോജ്യമാകുന്നു.

നിങ്ങൾക്ക് മൈക്രോസോഫ്റ്റ് ഫൗണ്ടറി പ്രോജക്ട് സജ്ജീകരിച്ചിട്ടില്ല എങ്കിൽ താഴെ കാണുന്ന ലോക്കൽ മോക്ക് മോഡൽ മാതൃകയെ പുറത്തെടുക്കുന്നു.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കുമായി ഏകീകരിക്കൽ

മുകളിൽ കാണിച്ചിരിക്കുന്ന `ReceiptedTool` റാപ്പർ ഫ്രെയിംവർക്ക്-സ്വതന്ത്രമാണ്. മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കിൽ നിർമ്മിച്ച ഏജന്റിൻ്റെ ഉള്ളിൽ ഇത് ഉപയോഗിക്കാൻ, റാപ്പുചെയ്‌ത ഫംഗ്ഷനെ ഒരു ടൂളായി രജിസ്റ്റർ ചെയ്യുക. ഒരു രൂപരേഖ (നിങ്ങൾ mock മാറ്റി യഥാർത്ഥ മൈക്രോസോഫ്റ്റ് ഫൗൻട്രി ടൂൾ രജിസ്ട്രേഷൻ ഉപയോഗിക്കും):

```python
# സംയോജന രൂപം കാണിക്കുന്ന പിസ്യൂഡോകോഡ്.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="നീയൊരു Contoso യാത്രാ ഏജന്റാണ് ...",
#     tools=[receipted_lookup],   # അട്ടകപ്പെടുത്തിയ ടൂൾ,_raw ഫങ്ഷൻ അല്ല
# )
# response = agent.run("ജൂണില് സിഡ്‌നി മുതൽ ലോസ് ആഞ്ചലസ് വരെ വിമാനങ്ങൾ കണ്ടെത്തുക.")
#
# # റൺ കഴിഞ്ഞ്, ഏജന്റ് നടത്തിയ എല്ലാ ടൂൾ കോൾക്കും ഒപ്പിട്ടിരിക്കുന്ന രജിസ്റ്റർ ഉണ്ടായിരിക്കും:
# audit_chain = receipted_lookup.receipts
```

ഏജന്റ് ഫ്രെയിംവർക്ക് രസീതുകളുടെ വിശദാംശങ്ങളെ അറിയേണ്ടതില്ല. റസീത്ത് ഒപ്പിടൽ ടൂൾ ചുറ്റുമാണ് റാപ്പ് ചെയ്യപ്പെട്ടിരിക്കുന്നത്, ഫ്രെയിംവർക്കിൽ ചേർത്തിട്ടല്ല. ഇതാണ് ഏജന്റിന്റെ പഴയ കോഡ് പുനഃരാഖ്യാനം ചെയ്യാതെ ഉണ്ടാക്കപ്പെട്ട കോഡിൽ出处 ചേർക്കാനുള്ള മാർഗം.


## സംഗ്രഹവും ഒരു വിപുലമായ ചലഞ്ചും

നിങ്ങളുടെ കൈവശം ഉണ്ട്:

- ഒരു Ed25519 കീ ജോഡി സൃഷ്ടിച്ചു.
- ഒരു ഏജന്റ് ടൂൾ കോളിന്റെ ഏറ്റു കെട്ടുകയും ഒപ്പിട്ടും.
- പൊതുവായ കീ മാത്രം ഉപയോഗിച്ച് ഓഫ്ലൈനിൽ ഏറ്റുവയ്ക്കല്‍ സ്ഥിരീകരിച്ചു.
- ഒരു ഏറ്റു ഭേദഗതി ചെയ്ത് സ്ഥിരീകരണം പരാജയപ്പെടുന്നത് കണ്ടു.
- മൂന്ന് ഏറ്റുക്കളുടെ ഹാഷ്-ചെയിൻ ചെയ്ത സംഖ്യ നിർമ്മിച്ചു.
- പരസ്പരത്തിലുള്ള ഭാഗം ഭേദഗതി ചെയ്ത് ഒപ്പം ഒപ്പിന്റെയും ചെയിൻ-ലിങ്ക് പരാജയപ്പെടുന്നതും കണ്ടു.
- ഒരു ടൂൾ ഫംഗ്ഷൻ സ്വയം ഒപ്പിട്ട കൊടുക്കലിൽ ഓട്ടോമാറ്റിക് ഏറ്റു ഒപ്പിടലുമായി പൊതിഞ്ഞു.

**വിപുലമായ ചലഞ്ച്.** `request_id` എന്ന ഫീൽഡ് (വിതരണ ട്രേസിംഗിന് UUID) ചേർത്ത് ഉണ്ടായെടുത്ത ഏറ്റു സ്കീമ വികസിപ്പിക്കുക. അതിൽ `make_receipt` അപ്ഡേറ്റ് ചെയ്ത് അവസാന വരെയുള്ള സ്ഥിരീകരണം ഉറപ്പാക്കുക. തുടർന്ന് ഒപ്പിട്ടതിനു ശേഷം ആ ഫീൽഡ് മാറ്റി സ്ഥിരീകരണം പരാജയപ്പെടുന്നത് സ്ഥിരീകരിക്കുക. ഇത് കാനോനിക്കൽ എൻകോഡിങ്ങിലെ ഓരോ ബൈറ്റും ഒപ്പുപ്രക്രിയയിൽ എന്തൊക്കെ സംഭാവനകൾ ചെയ്യുന്നു എന്ന് ആന്തരീകപ്പെടുത്താൻ നിങ്ങളെ പ്രേരിപ്പിക്കും.

**പ്രധാന പരിധി.** ഏറ്റുകൾ മൂന്നു കാര്യങ്ങൾ മാത്രമാണ് തെളിയിക്കുന്നത്: ഉറവിടം (ഈ കീ ഈ ഉള്ളടക്കം ഒപ്പിട്ടു), അതിന്റെ അഖണ്ഡത (ഒപ്പിടുമ്പോൾ നിന്ന് ഉള്ളടക്കം മാറിയിട്ടില്ല), ഒപ്പം ക്രമീകരണം (ഈ ഏറ്റു ആ ഏറ്റിനുശേഷം വന്നു). അവ ഏജന്റിന്റെ പ്രവർത്തനം ശരിയാണ്, `policy_id`-ൽ പറയുന്ന നയം വിലയിരുത്തപ്പെട്ടതാണോ, അല്ലെങ്കിൽ ഏജന്റ് എല്ലാ നിയമങ്ങളും പാലിച്ചിട്ടുണ്ടോ എന്നത് തെളിയിക്കാറില്ല. ഏറ്റുകൾ ഒരു അടിസ്ഥാനം ആണ്. ഗവർണൻസ് നിങ്ങൾ നിർമിക്കുന്ന സിസ്റ്റമാണ്.

ആ പരിധി മനസ്സിൽ വച്ചുകൊണ്ട് പാഠം README വീണ്ടും വായിക്കുക. ടീമുകൾ ഏറ്റുകളുമായി ചെയ്യാറുള്ള ഏറ്റവും സാധാരണ പിഴവ് "നമുക്ക് ഏറ്റുകൾ ഉണ്ട്" എന്നത് "നമുക്ക് ഗവർണൻസ് ഉണ്ട്" എന്ന് കരുതുകയാണ്. അത് ശരിയല്ല. ഏറ്റുകൾ ഏജന്റിന്റെ പ്രവൃത്തികളെ ഓഡിറ്റ് ചെയ്യാൻ സഹായിക്കുന്നു. അവ ശരിയായതാക്കുന്നില്ല.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
